# 04 — Results Summary

Consolidates every `outputs/best_model_task*/results.json` (written automatically
by `src/train.py`) into one comparison table, and lines it up against the
published EDOS baselines cited in the proposal (Table I).

## Setup: clone repo, restore every run's `results.json` from Drive

Unlike `03_explainability.ipynb`'s `copy_best_checkpoints` (which only restores the 6 final default checkpoints), this restores **every** run folder's `results.json` -- sweep variants included -- since the cells below need to see the full sweep to pick the best one. It skips the actual model weight files (multi-GB, and this notebook only reads metrics), so it's fast even with a lot of sweep runs backed up.

Safe to run even if you already ran `03_explainability.ipynb`'s setup cell in this session -- it just adds the `results.json` files it doesn't already have.

In [1]:
# Fresh session only -- skip if continuing directly from 02_training.ipynb or 03_explainability.ipynb
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/Mahekd/interpretable-nlp-sexism-detection.git
%cd interpretable-nlp-sexism-detection

import glob
import os
import shutil

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/EDOS_DATA/outputs_backup"

os.makedirs("outputs", exist_ok=True)
restored = 0
for src_path in glob.glob(os.path.join(DRIVE_BACKUP_DIR, "best_model_task*", "results.json")):
    run_dir = os.path.basename(os.path.dirname(src_path))
    dst_dir = os.path.join("outputs", run_dir)
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy(src_path, os.path.join(dst_dir, "results.json"))
    restored += 1

print(f"Restored {restored} results.json file(s) from {DRIVE_BACKUP_DIR}")
if restored == 0:
    print("Nothing found -- check DRIVE_BACKUP_DIR, or that 02_training.ipynb actually backed up to Drive.")

Mounted at /content/drive
Cloning into 'interpretable-nlp-sexism-detection'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 125 (delta 61), reused 78 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 6.38 MiB | 19.55 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/interpretable-nlp-sexism-detection
Restored 34 results.json file(s) from /content/drive/MyDrive/EDOS_DATA/outputs_backup


In [2]:
import glob
import json

import pandas as pd

rows = []
for path in glob.glob('outputs/best_model_task*/results.json'):
    with open(path) as f:
        r = json.load(f)
    rows.append({
        'task': r['task'],
        'model': r['model_name'],
        'run_name': r.get('run_name', 'default'),
        'augment': r['augment'],
        'lr': r.get('lr'),
        'batch_size': r.get('batch_size'),
        'epochs_run': r.get('epochs_run'),
        'dev_macro_f1': round(r['dev_macro_f1'], 4),
        'test_macro_f1': round(r['test_macro_f1'], 4),
    })

results_df = pd.DataFrame(rows).sort_values(['task', 'model', 'dev_macro_f1'], ascending=[True, True, False]).reset_index(drop=True)
results_df

,task,model,run_name,augment,lr,batch_size,epochs_run,dev_macro_f1,test_macro_f1
0,A,bert-base-uncased,lr2e-5_bs16,none,0.00002,16,8,0.8271,0.8195
1,A,bert-base-uncased,lr3e-5_bs32,none,0.00003,32,6,0.8250,0.8175
2,A,bert-base-uncased,lr2e-5_bs32,none,0.00002,32,5,0.8243,0.8198
3,A,bert-base-uncased,lr3e-5_bs16,none,0.00003,16,10,0.8165,0.8268
4,A,bert-base-uncased,default,none,0.00002,16,4,0.8134,0.8217
5,A,roberta-base,default,none,0.00002,32,4,0.8324,0.8283
6,A,roberta-base,lr2e-5_bs32,none,0.00002,32,4,0.8320,0.8321
7,A,roberta-base,lr3e-5_bs16,none,0.00003,16,10,0.8281,0.8244
8,A,roberta-base,lr2e-5_bs16,none,0.00002,16,4,0.8258,0.8231
9,A,roberta-base,lr3e-5_bs32,none,0.00003,32,2,0.8232,0.8150


## Best run per task + model (by dev macro-F1)

In [3]:
best_runs = (
    results_df
    .sort_values('dev_macro_f1', ascending=False)
    .groupby(['task', 'model'], as_index=False)
    .first()
    .sort_values(['task', 'model'])
    .reset_index(drop=True)
)
best_runs

,task,model,run_name,augment,lr,batch_size,epochs_run,dev_macro_f1,test_macro_f1
0,A,bert-base-uncased,lr2e-5_bs16,none,0.00002,16,8,0.8271,0.8195
1,A,roberta-base,default,none,0.00002,32,4,0.8324,0.8283
2,B,bert-base-uncased,default,none,0.00002,32,8,0.6453,0.5791
3,B,roberta-base,lr3e-5_bs16,none,0.00003,16,3,0.6567,0.5856
4,C,bert-base-uncased,default,none,0.00002,16,6,0.4732,0.4385
5,C,roberta-base,lr3e-5_bs16,none,0.00003,16,6,0.4482,0.4644


In [4]:
pivot = results_df.pivot_table(index=['model', 'augment'], columns='task', values='test_macro_f1')
pivot

task                                   A        B        C
model             augment                                 
bert-base-uncased backtranslate      NaN  0.57765      NaN
                  none           0.82106  0.57798  0.41546
roberta-base      backtranslate      NaN  0.59820      NaN
                  none           0.82458  0.60002  0.44770

## Published EDOS baselines (proposal Table I, macro-F1)

In [5]:
baselines = pd.DataFrame([
    {'system': 'Most Frequent Baseline', 'Task A': 0.431, 'Task B': 0.159, 'Task C': 0.032},
    {'system': 'DistilBERT Baseline',    'Task A': 0.780, 'Task B': 0.537, 'Task C': 0.314},
    {'system': 'DeBERTa-v3 Baseline',    'Task A': 0.824, 'Task B': 0.593, 'Task C': 0.317},
    {'system': 'Mahmoudi (BERT)',        'Task A': 0.830, 'Task B': 0.640, 'Task C': 0.470},
    {'system': 'Goldzycher (DeBERTa)',   'Task A': 0.859, 'Task B': 0.648, 'Task C': 0.449},
    {'system': 'Best SemEval System',    'Task A': 0.875, 'Task B': 0.720, 'Task C': 0.549},
]).set_index('system')
baselines

,Task A,Task B,Task C
system,,,
Most Frequent Baseline,0.431,0.159,0.032
DistilBERT Baseline,0.780,0.537,0.314
DeBERTa-v3 Baseline,0.824,0.593,0.317
Mahmoudi (BERT),0.830,0.640,0.470
Goldzycher (DeBERTa),0.859,0.648,0.449
Best SemEval System,0.875,0.720,0.549


## Faithfulness summary

In [6]:
import os

FAITHFULNESS_SUMMARY_PATH = "/content/drive/MyDrive/EDOS_DATA/explainability_results/explainability_summary.csv"

if os.path.exists(FAITHFULNESS_SUMMARY_PATH):
    faithfulness_summary = pd.read_csv(FAITHFULNESS_SUMMARY_PATH)
else:
    print(f"Not found yet: {FAITHFULNESS_SUMMARY_PATH}")
    print("Run the persistence cell at the end of 03_explainability.ipynb first, "
          "then re-run this cell.")
    faithfulness_summary = pd.DataFrame()

faithfulness_summary

,task,model,mean_comprehensiveness,mean_sufficiency,lime_shap_agreement
0,A,BERT-base,0.249105,0.232418,0.380000
1,A,RoBERTa-base,0.224622,0.106265,0.500000
2,B,BERT-base,0.445061,0.252653,0.510000
3,B,RoBERTa-base,0.273121,0.066777,0.595000
4,C,BERT-base,0.366516,0.199625,0.508182
5,C,RoBERTa-base,0.411008,0.324837,0.463636
